# Embedding Validation
This notebook tests the `FeatureProcessor` from `src/embedder.py` on the new Public and Private training tables.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add src to path
sys.path.append('../')
from src.embedder import FeatureProcessor

processor = FeatureProcessor()

/Users/ryanfue/Library/Caches/pypoetry/virtualenvs/smartcomps-16436_SS-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19501.35it/s]


## 1. Load Public Training Table

In [2]:
# Load data
data_path = "../data/processed/PUBLIC_training.parquet"

df = pd.read_parquet(data_path)
print(f"Loaded {len(df)} rows")
df.head()

Loaded 8050 rows


,ticker,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,sector,business_summary
0,NVDA,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,Technology,NVIDIA Corporation operates as a data center s...
1,GOOGL,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,Communication Services,Alphabet Inc. offers various products and plat...
2,AAPL,4.551953e+12,32.155922,28.454,159975997440,6.850700e+10,8.471100e+10,Technology,"Apple Inc. designs, manufactures, and markets ..."
3,MSFT,3.156524e+12,21.645250,17.113,184457003008,7.822800e+10,1.254320e+11,Technology,Microsoft Corporation develops and supports so...
4,AMZN,2.957284e+12,27.011812,18.974,155860992000,1.430890e+11,2.355400e+11,Consumer Cyclical,"Amazon.com, Inc. engages in the retail sale of..."


## 2. Test Embedding
Using `all-MiniLM-L6-v2` local model.

In [3]:
df_emb = processor.embed_summaries(df.head(10)) # Small sample

df_emb.head()

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.58it/s]


,ticker,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,sector,business_summary,nlp_0,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,NVDA,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,Technology,NVIDIA Corporation operates as a data center s...,-0.059150,...,-0.035018,-0.028591,0.014662,-0.092712,0.029643,0.026535,-0.013199,-0.065995,0.014612,0.018574
1,GOOGL,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,Communication Services,Alphabet Inc. offers various products and plat...,-0.059735,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
2,AAPL,4.551953e+12,32.155922,28.454,159975997440,6.850700e+10,8.471100e+10,Technology,"Apple Inc. designs, manufactures, and markets ...",-0.042323,...,-0.040436,0.024960,0.037074,-0.097955,0.022558,0.120378,0.046419,-0.064816,0.104943,0.013485
3,MSFT,3.156524e+12,21.645250,17.113,184457003008,7.822800e+10,1.254320e+11,Technology,Microsoft Corporation develops and supports so...,-0.014270,...,0.022693,0.035224,0.073339,-0.102653,0.079596,0.118757,-0.010114,-0.027585,0.031214,-0.044478
4,AMZN,2.957284e+12,27.011812,18.974,155860992000,1.430890e+11,2.355400e+11,Consumer Cyclical,"Amazon.com, Inc. engages in the retail sale of...",0.039800,...,0.040109,-0.000710,0.027882,-0.100945,0.072389,0.122982,-0.006967,-0.052227,0.004188,0.016085


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between first two companies
sim = cosine_similarity([df_emb.iloc[0].filter(like='nlp_')], [df_emb.iloc[1].filter(like='nlp_')])
print(f"Similarity between {df_emb.iloc[0]['ticker']} and {df_emb.iloc[1]['ticker']}: {sim[0][0]:.4f}")

Similarity between NVDA and GOOGL: 0.4038


## 3. Test Column Dropping
Ensuring that non-numeric columns are removed for ML readiness.

In [5]:
# Demonstrate dropping extra columns
df_ml = processor.drop_extra_columns(df_emb)
print(f"Columns after dropping: {df_ml.columns.tolist()[:10]}...")
print(f"Remaining columns: {len(df_ml.columns)}")

df_ml.head()

Columns after dropping: ['enterprise_value', 'forwardPE', 'ev_to_ebitda', 'ebitda', 'total_cash', 'total_debt', 'nlp_0', 'nlp_1', 'nlp_2', 'nlp_3']...
Remaining columns: 390


,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,nlp_0,nlp_1,nlp_2,nlp_3,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,-0.059150,-0.069233,-0.053639,-0.023783,...,-0.035018,-0.028591,0.014662,-0.092712,0.029643,0.026535,-0.013199,-0.065995,0.014612,0.018574
1,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,-0.059735,-0.116992,0.044080,-0.085275,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
2,4.551953e+12,32.155922,28.454,159975997440,6.850700e+10,8.471100e+10,-0.042323,-0.021482,-0.009602,-0.061581,...,-0.040436,0.024960,0.037074,-0.097955,0.022558,0.120378,0.046419,-0.064816,0.104943,0.013485
3,3.156524e+12,21.645250,17.113,184457003008,7.822800e+10,1.254320e+11,-0.014270,-0.056247,0.000315,-0.068629,...,0.022693,0.035224,0.073339,-0.102653,0.079596,0.118757,-0.010114,-0.027585,0.031214,-0.044478
4,2.957284e+12,27.011812,18.974,155860992000,1.430890e+11,2.355400e+11,0.039800,-0.067963,-0.031519,-0.043795,...,0.040109,-0.000710,0.027882,-0.100945,0.072389,0.122982,-0.006967,-0.052227,0.004188,0.016085


## 4. Verify Final Embedded Datasets
Verify the output of the full pipeline for both Public and Private engines.

In [7]:
public_embedded = "../data/processed/PUBLIC_embedded.parquet"
private_embedded = "../data/processed/PRIVATE_embedded.parquet"

if os.path.exists(public_embedded):
    df_pub = pd.read_parquet(public_embedded)
    print(f"Public Embedded Shape: {df_pub.shape}")
    display(df_pub.head(2))

if os.path.exists(private_embedded):
    df_priv = pd.read_parquet(private_embedded)
    print(f"Private Embedded Shape: {df_priv.shape}")
    display(df_priv.head(2))

Public Embedded Shape: (8050, 390)


,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,nlp_0,nlp_1,nlp_2,nlp_3,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,-0.059150,-0.069233,-0.053639,-0.023783,...,-0.035018,-0.028591,0.014662,-0.092712,0.029643,0.026535,-0.013199,-0.065995,0.014612,0.018574
1,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,-0.059735,-0.116992,0.044080,-0.085275,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426


Private Embedded Shape: (5751, 387)


,enterprise_value,employee_count,estimated_revenue,nlp_0,nlp_1,nlp_2,nlp_3,nlp_4,nlp_5,nlp_6,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,5.170628e+12,42000.0,2.534910e+11,-0.059150,-0.069233,-0.053639,-0.023783,-0.028156,-0.109184,0.015164,...,-0.035018,-0.028591,0.014662,-0.092712,0.029643,0.026535,-0.013199,-0.065995,0.014612,0.018574
1,4.609101e+12,194668.0,4.224980e+11,-0.059735,-0.116992,0.044080,-0.085275,-0.003429,0.052656,-0.020798,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
